## Homework: Crypto Spread Trading

### Data Preprocessing

In [1]:
import numpy as np
import polars as pl
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
def load_and_preprocess_single_date(
    exchange_name: str,
    date_str: str,
) -> pd.DataFrame:
    """
    Load and preprocess trade data for a single exchange and given date.

    Loads a parquet file, filters for trades (`rec_type == "T"`), converts relevant
    columns to appropriate types, sets the timestamp as the index, resamples
    to 1-second frequency (taking the last trade in each second), and returns
    only the `price` and `qty` columns as a pandas DataFrame.

    Note: The `.shift()` after resampling makes each row at time t represent the last trade 
    from the previous second (t-1). This ensures that the information at time t uses only data 
    that was known at the beginning of second t, rather than information from trades within t.

    Args:
        exchange_name (str): Name of the exchange (e.g., 'binance', 'OKX', 'Coinbase').
        date_str (str): Date string in the format 'YYYYMMDD'.

    Returns:
        pd.DataFrame: Preprocessed DataFrame with 'price' and 'qty' indexed by timestamp.
    """
    parquet_path = f"data/ETH-USDT/{exchange_name}/{date_str}.parquet"
    df_polars = pl.read_parquet(parquet_path)
    df_polars = df_polars.filter(pl.col("rec_type") == "T")
    df = df_polars.to_pandas()
    df["price"] = df["price"].astype(float)
    df["qty"] = df["qty"].astype(float)
    df = df.set_index("ts")
    df = df.resample("1s").last().shift()
    df = df[["price", "qty"]]
    return df

In [3]:
def load_multiple_exchanges_dates(exchange_names, dates):
    """
    Load and preprocess trade data for multiple exchanges and dates.

    Returns:
        dict: Keys are f"{exchange}_{date}" and values are preprocessed DataFrames.
    """
    return {
        f"{name}_{date}": load_and_preprocess_single_date(name, date)
        for name in exchange_names
        for date in dates
    }

exchange_names = ['Binance', 'Coinbase', 'OKX']
dates = ['20250524', '20250525']
exchange_date_dfs = load_multiple_exchanges_dates(exchange_names, dates)

In [4]:
def concatenate_exchange_dataframes(
    exchange_date_dfs: dict,
    exchanges: list = None
) -> dict:
    """
    Concatenate trade DataFrames for each exchange across all available dates.

    Args:
        exchange_date_dfs (dict): A dictionary with keys in the format 'Exchange_Date'
                                  and values as pandas DataFrames.
        exchanges (list, optional): List of exchange names to process. Defaults to
                                    ['Binance', 'Coinbase', 'OKX'].

    Returns:
        dict: Dictionary with keys as exchange names and values as concatenated
              and time-sorted DataFrames per exchange.
    """
    if exchanges is None:
        exchanges = ['Binance', 'Coinbase', 'OKX']

    concatenated_dfs = {}
    for exchange in exchanges:
        # Collect all DataFrames belonging to this exchange across all dates
        dfs = [
            df for key, df in exchange_date_dfs.items()
            if key.startswith(f"{exchange}_")
        ]
        if dfs:
            # Concatenate along the time index and sort chronologically
            concatenated_dfs[exchange] = pd.concat(dfs).sort_index()

    return concatenated_dfs

# Concatenate data for all exchanges, then assign each exchange's DataFrame to its own variable
exchange_concat_dfs = concatenate_exchange_dataframes(exchange_date_dfs)
binance = exchange_concat_dfs.get('Binance')
coinbase = exchange_concat_dfs.get('Coinbase')
okx = exchange_concat_dfs.get('OKX')

# Combine price columns from each exchange into a single DataFrame
price_panel = pd.concat([binance['price'].rename('binance_price'),
                        coinbase['price'].rename('coinbase_price'),
                        okx['price'].rename('okx_price')], axis=1)

# Forward fill the missing values and drop NaNs
price_panel = price_panel.ffill().dropna()

# Display the first few rows of the price panel
price_panel.head()

,binance_price,coinbase_price,okx_price
ts,,,
2025-05-24 00:00:07,2525.53,2525.57,2525.60
2025-05-24 00:00:08,2525.31,2525.75,2525.53
2025-05-24 00:00:09,2525.84,2525.75,2526.00
2025-05-24 00:00:10,2525.79,2525.75,2526.10
2025-05-24 00:00:11,2525.80,2525.75,2525.90


In [ ]:
# Plot the closing price of ETH-USDT from each exchange
series_specs = [
    ("Binance", "binance_price"),
    ("Coinbase", "coinbase_price"),
    ("OKX", "okx_price"),
]

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=[name for name, _ in series_specs],
    shared_yaxes=True,
)

for col_idx, (name, col_name) in enumerate(series_specs, start=1):
    fig.add_trace(
        go.Scatter(
            x=price_panel.index,
            y=price_panel[col_name],
            mode="lines",
            name=name,
            showlegend=False,
        ),
        row=1,
        col=col_idx,
    )
    fig.update_xaxes(title_text="Time", row=1, col=col_idx)

fig.update_yaxes(title_text="Price", row=1, col=1)
fig.update_layout(
    title_text="<span style='text-align:center; display:block;'>ETH-USDT Closing Price</span>", 
    title_x=0.5,
    height=450,
    width=1400,
)

fig.show()

### Spread Signals

#### Base Spreads

In [5]:
# A: Binance - Coinbase
sb_A = price_panel["binance_price"] - price_panel["coinbase_price"]

# B: Binance - OKX
sb_B = price_panel["binance_price"] - price_panel["okx_price"]

# C: Coinbase - OKX
sb_C = price_panel["coinbase_price"] - price_panel["okx_price"]

base_spreads = pd.DataFrame(
    {
        "Binance to Coinbase": sb_A,
        "Binance to OKX": sb_B,
        "Coinbase to OKX": sb_C,
    },
    index=price_panel.index,
)

# Display the first few rows of the base spreads
base_spreads.head()

,Binance to Coinbase,Binance to OKX,Coinbase to OKX
ts,,,
2025-05-24 00:00:07,-0.04,-0.07,-0.03
2025-05-24 00:00:08,-0.44,-0.22,0.22
2025-05-24 00:00:09,0.09,-0.16,-0.25
2025-05-24 00:00:10,0.04,-0.31,-0.35
2025-05-24 00:00:11,0.05,-0.10,-0.15


In [ ]:
# Plot the base spreads
cols = [
    "Binance to Coinbase",
    "Binance to OKX",
    "Coinbase to OKX",
]

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=cols,
    shared_yaxes=True,
)

for i, col in enumerate(cols, start=1):
    fig.add_trace(
        go.Scatter(
            x=base_spreads.index,
            y=base_spreads[col],
            mode="lines",
            showlegend=False,
        ),
        row=1,
        col=i,
    )
    fig.update_xaxes(title_text="Time", row=1, col=i)

fig.update_yaxes(title_text="Spread (USDT)", row=1, col=1)
fig.update_layout(
    title_text="Base Spreads",
    title_x=0.5,
    height=450,
    width=1400,
)
fig.show()

#### EMA and shifted spreads

We compute an exponential moving average (EMA) of each base spread using a **half-life of 3 hours**.
Because the data is 1-second regularized, half-life = 3 * 60 * 60 = 10800 seconds.

Shifted spread:
$$
s^{\mathrm{shift}}_t = s^b_t - a_t
$$
where $a_t$ is the exponential moving average (EMA) of the base spread $s^b_t$.


In [6]:
HALF_LIFE_SECONDS = 3 * 60 * 60

ema_spreads = base_spreads.ewm(halflife=HALF_LIFE_SECONDS, adjust=False).mean()
shifted_spreads = base_spreads - ema_spreads

shifted_spreads.head()

,Binance to Coinbase,Binance to OKX,Coinbase to OKX
ts,,,
2025-05-24 00:00:07,0.000000,0.000000,0.000000
2025-05-24 00:00:08,-0.399974,-0.149990,0.249984
2025-05-24 00:00:09,0.130017,-0.089985,-0.220002
2025-05-24 00:00:10,0.080012,-0.239969,-0.319981
2025-05-24 00:00:11,0.090006,-0.029967,-0.119974


In [ ]:
# Plot the shifted spreads
cols = [
    "Binance to Coinbase",
    "Binance to OKX",
    "Coinbase to OKX",
]

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=cols,
    shared_yaxes=True,
)

for i, col in enumerate(cols, start=1):
    fig.add_trace(
        go.Scatter(
            x=shifted_spreads.index,
            y=shifted_spreads[col],
            mode="lines",
            showlegend=False,
        ),
        row=1,
        col=i,
    )
    fig.update_xaxes(title_text="Time", row=1, col=i)

fig.update_yaxes(title_text="Shifted Spread (USDT)", row=1, col=1)
fig.update_layout(
    title_text="Shifted Spreads",
    title_x=0.5,
    height=450,
    width=1400,
)
fig.show()

#### Persistent Levels $p_S(t)$ and $p_L(t)$

We define *persistent levels* using order statistics computed over a rolling window of the shifted spread series $s_t$.
Let the rolling window length be $M$ (called `window` in code), and let $N$ be the order-statistic rank (1-indexed).

- **Persistent smallest level**:

$$
p_S(t) = \text{N-th smallest value in the last } M \text{ observations}.
$$

- **Persistent largest level**:

$$
p_L(t) = \text{N-th largest value in the last } M \text{ observations}.
$$

The output series $p_S(t)$ and $p_L(t)$ are then used in the trading rules (entry/exit/stop-loss) in the next section.

In [7]:
def rolling_nth_smallest(data, window, n):
    """
    Rolling N-th smallest over trailing `window` observations, ignoring NaNs.

    Accepts either:
    - pd.Series: returns pd.Series
    - pd.DataFrame: returns pd.DataFrame (computed column-wise)

    This corresponds to persistent level p_S(t) (N-th smallest in last M points).
    """
    _validate_rolling_order_stat_inputs(window, n)

    if window == 1:
        return data

    def nth_smallest_in_window(values):
        valid_values = values[~np.isnan(values)]
        if valid_values.size < n:
            return np.nan
        kth_index = n - 1
        return np.partition(valid_values, kth_index)[kth_index]

    return data.rolling(window=window).apply(nth_smallest_in_window, raw=True)


def rolling_nth_largest(data, window, n):
    """
    Rolling N-th largest over trailing `window` observations, ignoring NaNs.

    Accepts either:
    - pd.Series: returns pd.Series
    - pd.DataFrame: returns pd.DataFrame (computed column-wise)

    This corresponds to persistent level p_L(t) (N-th largest in last M points).
    """
    _validate_rolling_order_stat_inputs(window, n)

    if window == 1:
        return data

    def nth_largest_in_window(values):
        valid_values = values[~np.isnan(values)]
        if valid_values.size < n:
            return np.nan
        kth_index = valid_values.size - n
        return np.partition(valid_values, kth_index)[kth_index]

    return data.rolling(window=window).apply(nth_largest_in_window, raw=True)


def compute_persistent_levels(shifted_spread, window, n):
    """
    Compute both persistent levels for a shifted spread.

    Returns:
    - If input is Series: (pS_series, pL_series)
    - If input is DataFrame: (pS_df, pL_df)

    pS: N-th smallest in trailing window
    pL: N-th largest  in trailing window
    """
    pS = rolling_nth_smallest(shifted_spread, window=window, n=n)
    pL = rolling_nth_largest(shifted_spread, window=window, n=n)

    if isinstance(shifted_spread, pd.Series):
        pS = pS.rename("pS")
        pL = pL.rename("pL")
    else:
        pS = pS.add_suffix("_pS")
        pL = pL.add_suffix("_pL")

    return pS, pL


def _validate_rolling_order_stat_inputs(window, n):
    if window < 1:
        raise ValueError("window must be >= 1")
    if not (1 <= n <= window):
        raise ValueError("n must satisfy 1 <= n <= window")

In [8]:
pS_df, pL_df = compute_persistent_levels(shifted_spreads, window=1, n=1)

###  Entry and Exit Bands, Stop Loss Level

Rules:

- Enter **short spread** (size 1 ETH) when $$p_S(t) > g$$
- Enter **long spread** (size 1 ETH) when $$p_L(t) < -g$$
- Exit **short spread** when $$p_S(t) < j$$ or stop-loss when $$p_S(t) > \ell$$
- Exit **long spread** when $$p_L(t) > -j$$ or stop-loss when $$p_L(t) < -\ell$$
- At the end of the dataset, force exit at prevailing prices.
- Discard parameter sets that produce fewer than **5 trades per day**.
- Stop-loss triggers a **pause for remainder of the day** (no new trades).
- Trading costs: $\zeta \times \text{(gross traded value in USDT)}$ on entry and exit.
- Capital starts at **$80K**; if it hits **$40K**, close positions and stop trading permanently.